# Xuất CenterPoint (pillar 0.2 m, nuScenes) ra ONNX cho SAM3 Label Assist

1. Tải file `scripts/export_centerpoint_onnx.py` trong repo lên thư mục `/content` (biểu tượng thư mục ở cột trái).
2. Bấm **Runtime → Run all**. CPU là đủ, không cần GPU. Mất khoảng 5–10 phút.
3. Ô cuối tải về `centerpoint_pillar02_nus.onnx` (khoảng 90 MB). Mở app → Thành phần → "Dò vật 3D LiDAR (CenterPoint ONNX)" → Chọn file .onnx.

Colab dùng Python 3.13; mmcv 2.1.0 chỉ có bản cài sẵn cho Python 3.8–3.11, nên notebook tạo một Python 3.10 riêng ở `/content/py310`. Luôn gọi `/content/py310/bin/python`, không gọi `python`.

**Giấy phép:** code MMDetection3D Apache-2.0; trọng số học trên nuScenes → chỉ dùng phi thương mại / nghiên cứu.

In [ ]:
# Ô 1: Python 3.10 riêng bằng uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
!uv python install 3.10
!uv venv -p 3.10 /content/py310

In [ ]:
# Ô 2: thư viện (bản CPU). Cảnh báo "pip's dependency resolver ..." là bình thường
!uv pip install -p /content/py310/bin/python "torch==2.1.0" --index-url https://download.pytorch.org/whl/cpu
!uv pip install -p /content/py310/bin/python https://download.openmmlab.com/mmcv/dist/cpu/torch2.1.0/mmcv-2.1.0-cp310-cp310-manylinux1_x86_64.whl
!uv pip install -p /content/py310/bin/python "numpy<2" "opencv-python<4.11" "setuptools<70" "mmengine==0.10.4" "mmdet==3.3.0" "mmdet3d==1.4.0" "onnx==1.16.1" "onnxruntime==1.18.1"
![ -d mmdetection3d ] || git clone -q --depth 1 -b v1.4.0 https://github.com/open-mmlab/mmdetection3d.git

In [ ]:
# Ô 3: phải in "2.1.0+cpu 2.1.0 1.4.0"; thiếu file script thì báo trước
import os
assert os.path.exists("/content/export_centerpoint_onnx.py"), "Chưa tải export_centerpoint_onnx.py lên /content"
!/content/py310/bin/python -c "import torch, mmcv, mmdet3d; print(torch.__version__, mmcv.__version__, mmdet3d.__version__)"

In [ ]:
# Ô 4: xuất. Phải in "OK: ... [(1, 10, 128, 128), (1, 48, 128, 128), (1, 12, 128, 128)]"
%cd /content
!/content/py310/bin/python export_centerpoint_onnx.py --out centerpoint_pillar02_nus.onnx

In [ ]:
# Ô 5: tải file về máy (chỉ khi ô 4 in OK)
from google.colab import files
files.download("/content/centerpoint_pillar02_nus.onnx")